# Part 01: Literature Review  
## Project: Agastya – AI Contract Analysis System

This notebook explains **why** we study automatic contract analysis and **what** we plan to do in **Phase 1** of the Agastya project.

**What is Phase 1?**  
In Phase 1 we use **classical machine learning** only. That means methods like counting words, turning text into numbers (TF-IDF), and training models such as SVM. We **do not** use deep learning, BERT, or transformers in Phase 1. 

**What data do we use?**  
We work with **clause-level** examples. Each training example is: **one piece of contract text (a clause)** and **one label** (the type of clause). A common public dataset for this is **CUAD** (Contract Understanding Atticus Dataset). Example labels include: Payment, Termination, Confidentiality, Liability, and Governing Law.

## Problem statement

**Main question:** Can we use machine learning to **read and analyze legal contracts** in a useful way?

**Why this matters:**  
Contracts are long and written in difficult legal language. Important points—about **money (payment)**, **ending the contract (termination)**, **who is responsible if something goes wrong (liability)**, **secrets (confidentiality)**, **which country’s laws apply (governing law)**, and other duties—are often hidden inside big paragraphs.  

If a person reads the whole contract by hand, it takes a lot of time. Different people may also understand the same text differently.

**What Agastya tries to do (step by step):**

1. **Turn the contract into text on the computer**  
   If the contract is a scan or a photo, we first need real text (characters), not just a picture.

2. **Split the contract into clauses**  
   A **clause** is one logical piece of the contract (for example, one paragraph or one numbered point). We want the computer to work on **one clause at a time**.

3. **Guess the type of each clause**  
   For example: “This clause is about **payment**” or “This clause is about **termination**.” In Phase 1 we focus on this **classification** task.

4. **Later phases**  
   Things like **risk scoring** and **smart reasoning across many clauses** are planned for **Phase 2 and Phase 3**, not for Phase 1.

**Phase 1 goal in simple words:**  
Build a **clear baseline**: given the text of one clause, predict its **category**. We use simple, well-known ML tools so results are easy to explain and compare later.

## Why is contract understanding hard for a computer?

Here we list the main difficulties **in plain language**.

**1. Legal English is special**  
Words and phrases are formal. There are many technical terms. One part of the contract may say “see Section 5,” so meaning is **linked** across pages.

**2. The same idea can be written in many ways**  
Two clauses might mean almost the same thing but use **different words**. A system that only looks for one fixed phrase will **miss** many real cases.

**3. The document is not a simple table**  
Contracts have sections, bullet lists, schedules, and attachments. Layout changes from file to file. The computer must still get **clean text** out of it.

**4. Context matters**  
To fully understand one sentence, you sometimes need an **earlier definition** or **another clause**. Phase 1 still uses **one clause’s text** as input, but we should know that **full** understanding is harder.

**5. Some rules are not written directly**  
Sometimes what the parties must do is **implied**, not spelled out in one obvious sentence. That makes automatic reading harder.

**Why not only use fixed rules or keyword search?**  
Rules and keywords are easy to explain, but they **break** when wording changes.  

**What machine learning does instead:**  
We show the computer **many examples** where a human has already labeled each clause (e.g., “this is a payment clause”). The model **learns patterns** from those examples.  

**How we check if it works (Phase 1):**  
We report **F1-score**, **precision**, **recall**, and a **confusion matrix** (which categories get mixed up). 

- **Precision** (simple idea): Of all clauses the model called “payment,” how many were really payment?  
- **Recall** (simple idea): Of all true payment clauses, how many did the model find?  
- **F1-score**: A single number that balances precision and recall.  
- **Confusion matrix**: A table showing mistakes between categories.

## Document digitalization (OCR)

**The situation:**  
Many contracts are not born as Word files. They are **scanned pages**, **photos**, or **PDFs that are really pictures**. For those, the computer first sees **pixels**, not letters.

**What is OCR?**  
**OCR** stands for **Optical Character Recognition**. It means: **read text from an image** and output normal text (characters you can copy and search).

**Important: OCR is not the “AI lawyer”**  
OCR **only** answers: “What letters and words appear on this page?”  

It does **not** decide whether a clause is about payment or termination. It does **not** give legal advice.  

So in our pipeline, OCR is a **preparation step** (preprocessing). It turns a picture into **raw text**. After that, other steps (splitting clauses, TF-IDF, SVM) can run.

**Why OCR quality matters**  
If the scan is blurry, crooked, or handwritten badly, OCR makes **mistakes**. Wrong letters mean wrong words, and the classifier later gets **bad input**. That is why we may **fix** the image first (straighten the page, reduce noise) before OCR.

**Order of steps (big picture):**  

`Contract file (PDF or image) → optional image cleanup → OCR → plain text → split into clauses → turn text into numbers (TF-IDF) → machine learning model → predicted category`

## Classical machine learning for text (Phase 1 focus)

In Phase 1 we do **not** use neural networks or BERT. We use older but still strong methods that are **easy to describe**: first turn text into **numbers**, then train a **classifier**.

### Step A: Turn text into numbers

**Bag of Words (BoW) — simple idea**  
- We list words that appear in the training data.  
- For one clause, we make a **long list of counts**: how many times each word appears.  
- Word order is mostly ignored (we only see “which words showed up and how often”).  
- **N-grams** (optional): we can also count **short phrases** of 2 or 3 words in a row, so a little bit of order is kept (e.g., “late payment” as one piece).

**TF-IDF — simple idea**  
- **TF** = how important a word is **inside this clause** (often related to how often it appears there).  
- **IDF** = down-weight words that appear in **almost every** contract (like very common legal filler) because they do not help tell categories apart.  
- **Together**, TF-IDF gives a **numeric vector** for each clause. Many entries are zero (sparse), which is normal for text.

**Why this fits Agastya Phase 1**  
Each clause is usually **one chunk of text**, not a whole book. TF-IDF is a **standard baseline** for that kind of task.

### Step B: Train a classifier on those vectors

**Naive Bayes**  
- Uses probability ideas: “Given these words, how likely is each category?”  
- It makes a **simplifying assumption** (words are treated as somewhat independent), but it is **fast** and often used as a **baseline** to compare against.

**Support Vector Machine (SVM)**  
- Tries to draw a **boundary** in the high-dimensional space that **best separates** categories.  
- Works well when inputs are **sparse** and **high-dimensional**, which is exactly what TF-IDF looks like.  

- Main features: **TF-IDF** (with **n-grams** as needed).  
- **SVM** = main model.  
- **Naive Bayes** = extra baseline.  
- **No** deep learning, **no** transformers, **no** BERT in Phase 1.

## Limitations of classical machine learning

TF-IDF + SVM (or Naive Bayes) are **honest** tools: we know what they do. But they also have **clear limits** when we talk about **meaning** in legal text.

**1. Weak handling of “same meaning, different words”**  
The model mostly sees **exact words** (and n-grams). If two clauses use **different words** for the same legal idea, the model may treat them as unrelated unless those words appeared often enough in training.

**2. Word order is only partly used**  
Unless we add n-grams, “not liable” and mixing of negatives can be hard. Even with n-grams, **long** grammatical structure across a whole long sentence is **not** modeled deeply.

**3. Very long clauses**  
If one “clause” block is huge, the vector mixes **many topics** together. Then the label (one category) may not match the text well. **Good splitting** of clauses matters a lot.

**4. One clause at a time**  
The vector for **one** clause does not carry **direct** information about **another** clause (for example, how payment rules relate to termination). Real contracts are **connected**; classical Phase 1 classification usually **ignores** those links.

**Why we still do Phase 1**  
- It is **required** by the project rules for now.  
- It gives a **simple baseline** we can measure with F1, precision, recall, and a confusion matrix.  
- When Phase 2 adds richer models, we can **show** how much we gained instead of only claiming it.

This matches what evaluators look for: a **clear problem**, **clear method**, **clear metrics**, and **honest limits** (theoretical rigor—not only buzzwords).

## Proposed approach (Phase 1)

Phase 1 is the full path from a **file** to a **predicted clause type**, using **only** classical ML.

**Pipeline in one line:**

```
Document → OCR → split into clauses → TF-IDF (numbers) → classifier → predicted category
```

**What each step means:**

1. **OCR and preprocessing**  
   Get **plain text** from PDFs or images. Fix the image first if needed (straighten, reduce noise) so OCR errors are smaller.

2. **Clause segmentation**  
   Break the full contract text into **chunks** that match how labels were created (for example, one CUAD-style clause per row).

3. **Feature engineering (main: TF-IDF)**  
   Turn each clause into a **vector of numbers**. Use **n-grams** so short phrases count, not only single words. Optional extras from `project.md`: clause length, keywords, etc.

4. **Models**  
   - **Naive Bayes**: simple baseline, fast to train.  
   - **SVM**: **main** model for Phase 1.

5. **Evaluation**  
   Report **F1-score** (main), **precision**, **recall**, and a **confusion matrix** so we see **which categories** the model confuses.

**Why we skip deep learning here**  
Phase 1 **on purpose** does **not** use BERT or transformers. Later, when we add them, we can say: “The gain came from the new method,” not from mixing many changes at once.

## Future scope

The project is split into **three phases** so we improve **step by step** and can **report** each stage clearly (this also matches typical coursework expectations: clear story, clear metrics).

**Phase 2 — Deep learning**  
We can add models like **BERT** (or legal versions of BERT). In simple terms: the model builds a **richer internal representation** of the sentence, which often helps with **different wordings** and **longer context** **inside** one clause.

**Phase 3 — Hybrid system**  
We can combine:
- outputs from **classical** and/or **neural** models, and  
- **extra logic** for **risk** and **consistency** (for example: checking how one clause relates to another).

Phase 1 is not “wasted work.” It gives **numbers** and **error patterns**. Phase 2 and 3 should be **justified** by those results—for example, types of mistakes we see in the confusion matrix—not only because a tool is popular.

## Summary (quick reference)

| Topic | What it does in Agastya (simple) |
|--------|----------------------------------|
| OCR | Turns **pictures/scans** into **text**. Not a classifier. |
| BoW / TF-IDF | Turns **clause text** into **numbers** for the model. Use **n-grams** when useful. |
| Naive Bayes | **Baseline** classifier (easy and fast). |
| SVM | **Main** Phase 1 classifier. |
| Limits of classical ML | Explains **why** Phase 2/3 may add stronger models. |
| CUAD-style data | Each row: **one clause’s text** + **one label** (category). |

This notebook is **explanation only**. Other notebooks will **load data**, **train** models, and **print** F1, precision, recall, and confusion matrix under the **same Phase 1 rules**.